# 04 — Synthetic Model Training & Comparison

> ⚠️ **SYNTHETIC DATA — NOT REAL SURVEY RESPONSES**

**Inputs:**
- `data/synthetic/processed/X_features_synthetic.csv` (1000 × 38)
- `data/synthetic/processed/y_target_synthetic.csv` (1000,)

**What this notebook does:**
1. Loads synthetic feature matrix + target
2. Stratified train/test split (80/20)
3. StandardScaler fitted on train only
4. Trains 7 classifiers with 5-fold CV
5. Saves results and best model

**Purpose:** test whether the ML pipeline scales from real n=39 to
synthetic n=1000. Results are compared with the real-data pipeline
in notebook 05.

In [1]:
import pandas as pd
import numpy as np

X = pd.read_csv("../data/synthetic/processed/X_features_synthetic.csv")
y = pd.read_csv("../data/synthetic/processed/y_target_synthetic.csv").squeeze("columns")

print("X shape:", X.shape)
print("y shape:", y.shape)
print()
print("Target distribution:")
print(y.value_counts())

X shape: (1000, 38)
y shape: (1000,)

Target distribution:
stress_level
Moderate    342
Low         332
High        326
Name: count, dtype: int64


## 1. Train/Test Split (Stratified 80/20)

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)
print()
print("Train class distribution:")
print(y_train.value_counts())
print()
print("Test class distribution:")
print(y_test.value_counts())

Train size: (800, 38)
Test size : (200, 38)

Train class distribution:
stress_level
Moderate    274
Low         265
High        261
Name: count, dtype: int64

Test class distribution:
stress_level
Moderate    68
Low         67
High        65
Name: count, dtype: int64


## 2. Feature Scaling

In [3]:
from sklearn.preprocessing import StandardScaler
import os
import joblib

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaled. Shapes:", X_train_scaled.shape, X_test_scaled.shape)

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler_synthetic.pkl")
print("Saved: ../models/scaler_synthetic.pkl")

Scaled. Shapes: (800, 38) (200, 38)
Saved: ../models/scaler_synthetic.pkl


## 3. Baseline Model

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
acc_scores = cross_val_score(baseline, X_train_scaled, y_train, cv=cv, scoring="accuracy")
f1_scores  = cross_val_score(baseline, X_train_scaled, y_train, cv=cv, scoring="f1_macro")

print("Baseline (most_frequent):")
print(f"  Accuracy : {acc_scores.mean():.3f} (±{acc_scores.std():.3f})")
print(f"  F1 macro : {f1_scores.mean():.3f} (±{f1_scores.std():.3f})")

Baseline (most_frequent):
  Accuracy : 0.342 (±0.002)
  F1 macro : 0.170 (±0.001)


## 4. Train 7 Models with 5-Fold CV

In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc  = label_encoder.transform(y_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)":           SVC(kernel="rbf", random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":         GaussianNB(),
    "XGBoost":             XGBClassifier(
                               n_estimators=200, max_depth=3, learning_rate=0.1,
                               eval_metric="mlogloss",
                               random_state=42, verbosity=0
                           ),
}

int_label_models = {"XGBoost"}

results = []
for name, model in models.items():
    y_tr = y_train_enc if name in int_label_models else y_train

    acc = cross_val_score(model, X_train_scaled, y_tr, cv=cv, scoring="accuracy")
    f1  = cross_val_score(model, X_train_scaled, y_tr, cv=cv, scoring="f1_macro")

    results.append({
        "Model":    name,
        "Acc Mean": acc.mean(),
        "Acc Std":  acc.std(),
        "F1 Mean":  f1.mean(),
        "F1 Std":   f1.std(),
    })

    print(f"{name:22s}  Acc = {acc.mean():.3f} (±{acc.std():.3f})   "
          f"F1m = {f1.mean():.3f} (±{f1.std():.3f})")

print()
results_df = pd.DataFrame(results).sort_values("F1 Mean", ascending=False).reset_index(drop=True)
print("Ranked by F1 macro:")
print(results_df.round(3).to_string(index=False))

Logistic Regression     Acc = 0.738 (±0.031)   F1m = 0.737 (±0.030)
Decision Tree           Acc = 0.707 (±0.024)   F1m = 0.707 (±0.022)
Random Forest           Acc = 0.743 (±0.027)   F1m = 0.744 (±0.025)
SVM (RBF)               Acc = 0.724 (±0.014)   F1m = 0.724 (±0.013)
KNN (k=5)               Acc = 0.594 (±0.041)   F1m = 0.588 (±0.039)
Naive Bayes             Acc = 0.679 (±0.051)   F1m = 0.669 (±0.076)
XGBoost                 Acc = 0.736 (±0.018)   F1m = 0.736 (±0.017)

Ranked by F1 macro:
              Model  Acc Mean  Acc Std  F1 Mean  F1 Std
      Random Forest     0.742    0.027    0.744   0.025
Logistic Regression     0.738    0.031    0.737   0.030
            XGBoost     0.736    0.018    0.736   0.017
          SVM (RBF)     0.724    0.014    0.724   0.013
      Decision Tree     0.707    0.024    0.707   0.022
        Naive Bayes     0.679    0.051    0.669   0.076
          KNN (k=5)     0.594    0.041    0.588   0.039


## 5. Save Results & Best Model

In [6]:
import os
import joblib
import json

os.makedirs("../reports", exist_ok=True)

results_df.to_csv("../reports/model_comparison_synthetic.csv", index=False)
print("Saved: ../reports/model_comparison_synthetic.csv")

best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_model.fit(X_train_scaled, y_train)

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/best_model_synthetic.pkl")

metadata = {
    "model_name": best_name,
    "cv_f1_macro_mean": float(results_df.iloc[0]["F1 Mean"]),
    "cv_f1_macro_std":  float(results_df.iloc[0]["F1 Std"]),
    "cv_accuracy_mean": float(results_df.iloc[0]["Acc Mean"]),
    "trained_on": f"X_train_scaled (n={X_train.shape[0]}, p={X_train.shape[1]})",
    "test_set": f"held out (n={X_test.shape[0]})",
    "random_state": 42,
    "classes": list(y_train.unique()),
    "data_type": "synthetic",
    "note": "LLM-generated data. Findings are pipeline-validation only.",
}
with open("../models/best_model_synthetic_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Best model: {best_name}")
print("Saved: ../models/best_model_synthetic.pkl")
print("Saved: ../models/best_model_synthetic_metadata.json")

Saved: ../reports/model_comparison_synthetic.csv
Best model: Random Forest
Saved: ../models/best_model_synthetic.pkl
Saved: ../models/best_model_synthetic_metadata.json


## Summary

- ✅ Loaded synthetic feature matrix (1000 × 38)
- ✅ Stratified 80/20 split → 800 train / 200 test
- ✅ StandardScaler fitted on train only
- ✅ Baseline established
- ✅ 7 models trained with 5-fold CV
- ✅ Best model saved

**Next:** `05_real_vs_synthetic_comparison.ipynb` — the money shot